# Notebook 03 - Data Augmentation

Extracts the reef-safe sunscreen subset from the greenwashing topic, 
then generates synthetic Reddit documents via Claude API to augment 
the corpus for downstream sentiment analysis and NER.

**Input:** `../data/processed/df_greenwashing_topic1.csv`  
**Output:** `../data/processed/df_sunscreen_augmented.csv`

In [1]:
import pandas as pd
import re
import ast


In [2]:
# load greenwashing subset
df_greenwashing = pd.read_csv("../data/processed/df_greenwashing_topic1.csv", parse_dates=["Timestamp"])
print(f"Shape: {df_greenwashing.shape}")

Shape: (3602, 12)


## 1. Sunscreen Subset Extraction

Keyword filtering identifies documents that mention reef-safe sunscreen brands, 
chemical UV filters, or related terms within the greenwashing-dominant topic 
(Topic 1, n=3,602).

In [3]:
# filter for sunscreen-related keywords
keywords = (
    'sunscreen|sun screen|suncream|sun cream|sunblock|sun block|'
    'reef|coral|oxybenzone|octinoxate|reef-safe|'
    'banana boat|hawaiian tropic|stream2sea|'
    'spf|mineral sunscreen|chemical sunscreen'
)

mask = df_greenwashing['Text'].str.contains(keywords, case=False, na=False)
df_sunscreen = df_greenwashing[mask].copy()

print(f"Sunscreen documents: {len(df_sunscreen)}")
print(f"Posts: {len(df_sunscreen[df_sunscreen['content_type']=='post'])}")
print(f"Comments: {len(df_sunscreen[df_sunscreen['content_type']=='comment'])}")

Sunscreen documents: 8
Posts: 2
Comments: 6


In [4]:
# save the sunscreen-specific dataset
df_sunscreen.to_csv("../data/processed/df_sunscreen_real.csv", index=False)
print("✓ Saved!")

✓ Saved!


## 2. Synthetic Data Generation

The real sunscreen subset contains only 8 documents - insufficient
for sentiment analysis and NER.

Synthetic Reddit posts and comments were generated using the Claude API
to augment the dataset. Synthetic data covers:
- ACCC Federal Court case (July 2025) against Edgewell Personal Care
  (Banana Boat, Hawaiian Tropic) for greenwashing "reef friendly" claims
- oxybenzone and octinoxate harm to coral reefs
- reef-safe label greenwashing
- brand-specific mentions (greenwashing vs genuinely reef-safe brands)

Sentiment distribution: ~40% negative, ~25% positive,
~25% neutral, ~10% sarcastic.

260 synthetic documents were generated in 5 batches, with varied
themes and sentiment emphasis per batch to avoid repetition.

In [5]:
# batch 1
batch_1 = [
    {
        "Post_id": "t3_k9m2xp",
        "Timestamp": "14/07/2025 9:23",
        "Text": "ACCC just dropped the hammer on Banana Boat and Hawaiian Tropic - Federal Court proceedings for greenwashing!! Finally someone is doing something about these 'reef friendly' lies. Edgewell Personal Care has been slapping that label on products that contain oxybenzone and octinoxate for YEARS. These chemicals are literally banned in Hawaii and Palau because of how badly they damage coral reefs. I switched to Stream2Sea two years ago after doing my own research but the fact that millions of people trusted these brands makes me sick. Hope the ACCC wins this one and sets a precedent. About time consumer protection law caught up with this nonsense.",
        "Score": 87,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_ax3f7q",
        "Timestamp": "14/07/2025 10:45",
        "Text": "omg I literally have a bottle of banana boat 'reef friendly' spf 50 on my shelf rn. throwing it in the bin",
        "Score": 312,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_bz8w1n",
        "Timestamp": "14/07/2025 11:02",
        "Text": "The ACCC case was announced July 2025 - for those who haven't read it, they're specifically targeting the 'reef friendly' label as misleading under Australian consumer law. This is a federal court proceeding not just a fine, so it's pretty serious.",
        "Score": 156,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_p2n8vr",
        "Timestamp": "15/07/2025 8:14",
        "Text": "Hawaiian Tropic 'reef friendly' - a love story. Chapter 1: marketing team puts a cute little turtle on the bottle. Chapter 2: product contains chemicals literally toxic to coral polyps. Chapter 3: ACCC files Federal Court proceedings. Chapter 4: brands act surprised. The end. Honestly I'd laugh if it wasn't so infuriating. I work in marine conservation and we've been screaming about this for years. Nobody listened until a government body finally got involved. Raw Elements and Thinksport have been doing actual reef-safe formulations this whole time but they don't have the marketing budget to compete with Edgewell.",
        "Score": 74,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_cd9r4s",
        "Timestamp": "15/07/2025 9:30",
        "Text": "lmao 'reef friendly' while containing oxybenzone. peak capitalism right there",
        "Score": 489,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_dw5k2m",
        "Timestamp": "15/07/2025 12:17",
        "Text": "Does anyone know what penalty Edgewell Personal Care could actually face from the ACCC proceedings? Like are we talking a slap on the wrist fine or something with actual teeth?",
        "Score": 78,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_eh4t9j",
        "Timestamp": "15/07/2025 13:55",
        "Text": "Under Australian consumer law the ACCC can seek substantial civil penalties - for large companies it can be in the millions. Federal Court proceedings mean this is serious, not just a warning letter. They typically also seek injunctions and corrective advertising orders.",
        "Score": 203,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_q7h3wc",
        "Timestamp": "16/07/2025 7:45",
        "Text": "I've been buying Hawaiian Tropic for my kids for years specifically because of the reef friendly label. I thought I was doing the right thing for the environment. I feel genuinely stupid and betrayed. This isn't a small thing - I was taking my kids to the Great Barrier Reef and specifically chose this brand thinking we weren't contributing to reef damage. The ACCC case confirms what environmental scientists have been saying forever. Edgewell knowingly used misleading marketing to charge premium prices to people who cared about the environment. That's not a mistake, that's a deliberate business decision.",
        "Score": 93,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_fv2u8b",
        "Timestamp": "16/07/2025 9:12",
        "Text": "same. bought banana boat reef friendly every summer for like 4 years. feel so dumb",
        "Score": 267,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_gx6y5r",
        "Timestamp": "16/07/2025 10:40",
        "Text": "You're not dumb, you trusted a label that should have been truthful. The fault is entirely with Edgewell for putting it there in the first place.",
        "Score": 445,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_hk1p3n",
        "Timestamp": "16/07/2025 14:22",
        "Text": "Stream2Sea is genuinely reef safe btw if anyone is looking for alternatives. they actually test their products on coral larvae and publish the results. wild concept i know",
        "Score": 334,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_r5m9lk",
        "Timestamp": "17/07/2025 8:30",
        "Text": "Can we talk about how the 'reef friendly' label on Banana Boat and Hawaiian Tropic products had absolutely zero legal definition behind it? There was no standard, no certification, no independent verification. Edgewell just... put it on there. And now the ACCC is taking them to Federal Court over it which honestly should have happened much sooner. This is why we need mandatory standards for environmental claims not voluntary ones. The EU is actually ahead on this with their Green Claims Directive. Australia and the US need to catch up fast before every product on the shelf has meaningless eco labels.",
        "Score": 68,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_ij7a6w",
        "Timestamp": "17/07/2025 10:15",
        "Text": "the EU green claims directive is genuinely good policy. companies have to substantiate environmental claims BEFORE making them not after they get caught",
        "Score": 189,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_jq8f2x",
        "Timestamp": "17/07/2025 11:50",
        "Text": "Ah yes, 'reef friendly' - defined as 'friendly in the same way a bulldozer is friendly to a sandcastle'",
        "Score": 412,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_kn3s7d",
        "Timestamp": "17/07/2025 15:33",
        "Text": "I work at a dive shop and we've been refusing to stock banana boat and hawaiian tropic for 3 years because of this exact issue. customers always ask why and now i can just send them the accc announcement",
        "Score": 276,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_s4k7mj",
        "Timestamp": "18/07/2025 9:05",
        "Text": "ACCC v Edgewell Personal Care - what you need to know. The Australian Competition and Consumer Commission commenced Federal Court proceedings in July 2025 against Edgewell Personal Care, the company that makes Banana Boat and Hawaiian Tropic sunscreens. The core allegation is that the 'reef friendly' label on their products is misleading conduct under the Australian Consumer Law. The products in question contain chemical UV filters that scientific research has linked to coral bleaching and toxicity in marine organisms. The ACCC is seeking penalties and orders requiring Edgewell to remove or correct the claims. This follows years of advocacy from environmental groups and scientists. Worth following this case closely as the outcome could reshape how environmental claims are regulated in Australia.",
        "Score": 81,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_lm5c9p",
        "Timestamp": "18/07/2025 10:22",
        "Text": "thank you for this writeup, actually really helpful. sharing with my sustainability group",
        "Score": 94,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_mo2b4q",
        "Timestamp": "18/07/2025 12:45",
        "Text": "So basically Edgewell said 'reef friendly' because it sounds nice not because it means anything. got it.",
        "Score": 367,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_np9g1v",
        "Timestamp": "18/07/2025 14:08",
        "Text": "Thinksport SPF 50 has been my go-to for years. uses zinc oxide only, no nasty chemical filters. not as cheap as banana boat but at least it doesnt lie about being reef safe lol",
        "Score": 223,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_t6p1nw",
        "Timestamp": "19/07/2025 8:50",
        "Text": "Hot take: the ACCC case against Edgewell Personal Care is the tip of the iceberg. How many other sunscreen brands are slapping 'reef safe' or 'reef friendly' on their products with zero scientific backing? I did a quick check of the sunscreen aisle at my local chemist and counted at least 6 different brands using similar claims with NO ingredient transparency about which filters they use. If you contain oxybenzone, octinoxate, octocrylene or homosalate you are NOT reef safe full stop. The science on this is not seriously disputed anymore. The ACCC should be looking at every single brand making these claims not just Banana Boat and Hawaiian Tropic.",
        "Score": 72,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_or4d8h",
        "Timestamp": "19/07/2025 9:45",
        "Text": "octocrylene is the sneaky one - a lot of 'reef safe' products removed oxybenzone but kept octocrylene which is also harmful. always check the full ingredient list",
        "Score": 298,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_pw7j5t",
        "Timestamp": "19/07/2025 11:20",
        "Text": "imagine having the audacity to put a picture of healthy coral on your bottle while the contents actively bleach coral. this is the banana boat experience",
        "Score": 387,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_qx1m6u",
        "Timestamp": "19/07/2025 13:55",
        "Text": "The ACCC filing is publicly available if anyone wants to read the actual legal documents. Its pretty detailed about which specific products and claims are being challenged.",
        "Score": 142,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ry5n2s",
        "Timestamp": "20/07/2025 8:17",
        "Text": "my daughter came home from school yesterday talking about the banana boat thing because her teacher mentioned it in science class. at least some good is coming out of this case - its becoming a teachable moment about greenwashing",
        "Score": 176,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_u3w8ck",
        "Timestamp": "20/07/2025 9:30",
        "Text": "I used to defend Banana Boat when people online said it wasn't actually reef friendly. I said 'they have studies, they wouldn't just make it up.' I was wrong and I feel like an idiot. The ACCC proceedings have confirmed what environmental advocates were saying all along. Edgewell Personal Care was using 'reef friendly' as a pure marketing tactic aimed at environmentally conscious consumers, without the science to back it up. I've switched to Raw Elements which is genuinely mineral-based and has proper environmental certifications. Never going back to these big brands for sunscreen. The betrayal of trust is real.",
        "Score": 65,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_sz6k3b",
        "Timestamp": "20/07/2025 10:48",
        "Text": "Raw Elements is legit good. the packaging is also recycled metal tin which is infinitely better than plastic tubes. switched years ago and never looked back",
        "Score": 211,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ta9h7f",
        "Timestamp": "20/07/2025 12:33",
        "Text": "Ah yes the ol' 'put a turtle on it and call it reef friendly' strategy. Worked great until the ACCC showed up.",
        "Score": 456,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ub3r1p",
        "Timestamp": "21/07/2025 9:05",
        "Text": "does anyone know if theres a class action in the works? people who specifically bought these products for the reef friendly claim should be able to recover something imo",
        "Score": 134,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_vc7s4n",
        "Timestamp": "21/07/2025 10:22",
        "Text": "ACCC proceedings are separate from private class actions - the ACCC acts in the public interest. A private class action is theoretically possible but you'd need to show individual loss which is harder. The ACCC penalties go to consolidated revenue not back to consumers unfortunately.",
        "Score": 189,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_v9l5qm",
        "Timestamp": "21/07/2025 8:45",
        "Text": "Let's be real about what 'reef friendly' on Banana Boat and Hawaiian Tropic actually meant: absolutely nothing. It was a made-up marketing claim designed to extract more money from consumers who care about the environment. There is no industry standard for 'reef friendly'. There is no independent testing requirement. There is no certification body. Edgewell just put words on a bottle and hoped nobody would look too closely. The ACCC looked closely. Now we're in Federal Court. And honestly? Good. This is exactly what consumer protection law is for. Companies should not be able to weaponize environmental concern against the very consumers who hold it.",
        "Score": 78,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_wd2t8g",
        "Timestamp": "21/07/2025 14:15",
        "Text": "weaponize environmental concern is such a good phrase for this. saving that",
        "Score": 267,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_xe5u3k",
        "Timestamp": "22/07/2025 8:30",
        "Text": "Hawaiian Tropic built their whole brand on tropical paradise vibes and then actively contributing to destroying tropical ecosystems. the cognitive dissonance required is staggering",
        "Score": 341,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_yf8v6r",
        "Timestamp": "22/07/2025 10:55",
        "Text": "Following the ACCC v Edgewell case closely as someone who works in marine biology. The science on oxybenzone and coral is solid - it induces bleaching at very low concentrations. These claims were always indefensible from a scientific standpoint.",
        "Score": 228,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_w2g4rn",
        "Timestamp": "22/07/2025 9:10",
        "Text": "Petition to ban the phrase 'reef friendly' unless a product has independently verified mineral-only UV filters and third party testing. The ACCC case is great but we need systemic change. The problem isn't just Edgewell and their Banana Boat / Hawaiian Tropic lines, it's the entire lack of regulatory framework around environmental claims on personal care products. Stream2Sea and Raw Elements prove it CAN be done properly - these companies actually test on coral larvae, they actually publish their data, they actually use only non-nano zinc oxide as a filter. Why should companies that do the right thing compete against brands that just lie?",
        "Score": 61,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_zg1w9s",
        "Timestamp": "22/07/2025 11:30",
        "Text": "upvoted for Stream2Sea mention - they're tiny and deserve more recognition. genuinely transparent company",
        "Score": 154,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ah4x2t",
        "Timestamp": "23/07/2025 9:00",
        "Text": "ngl i didn't even know what oxybenzone was until this whole accc thing blew up. now i'm that person at the chemist reading every ingredient list like a maniac",
        "Score": 387,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_bi6y5u",
        "Timestamp": "23/07/2025 11:15",
        "Text": "Being that person is actually correct behaviour though. Companies have shown they will lie on labels if it helps sales. Checking ingredients is the only defence consumers have until regulation catches up.",
        "Score": 199,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_x8h6pj",
        "Timestamp": "23/07/2025 8:40",
        "Text": "The thing that gets me about the ACCC case against Edgewell Personal Care is that 'reef friendly' wasn't buried in fine print. It was a central selling point. It was on the front of the bottle in big friendly letters. It was part of their advertising campaigns. This wasn't a minor ambiguous claim - it was a deliberate, prominent marketing strategy aimed specifically at people who wanted to make ethical consumer choices. That's what makes it so egregious. They targeted the most conscientious consumers. I hope the Federal Court penalties reflect how calculating and deliberate this was.",
        "Score": 69,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_cj9z3v",
        "Timestamp": "23/07/2025 12:40",
        "Text": "this is a really good point - they weren't just passively misleading, they were TARGETING eco-conscious consumers. premium price point, specific marketing. thats calculated",
        "Score": 243,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_dk3a7w",
        "Timestamp": "24/07/2025 8:55",
        "Text": "Edgewell about to discover that 'reef friendly' has a legal definition now (it's whatever the ACCC says it is in Federal Court)",
        "Score": 378,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_el7b4x",
        "Timestamp": "24/07/2025 10:30",
        "Text": "For those asking about genuinely reef safe alternatives: look for products with non-nano zinc oxide as the ONLY active ingredient. Non-nano means the particles are too large to be absorbed by coral. Brands like Thinksport, Raw Elements and Stream2Sea all meet this standard. Avoid anything with oxybenzone, octinoxate, octocrylene, homosalate, octisalate.",
        "Score": 312,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_y1j9qk",
        "Timestamp": "24/07/2025 9:20",
        "Text": "Quick reminder that the Great Barrier Reef is already dealing with mass bleaching events due to climate change and warming ocean temperatures. The LAST thing it needs is chemical UV filters in the water from tourists who thought they were using reef-safe sunscreen. The Edgewell greenwashing case isn't just about consumer fraud - it has actual environmental consequences. People going to the GBR with Banana Boat 'reef friendly' sunscreen aren't just getting ripped off, they're potentially contributing to reef damage while thinking they're protecting it. The irony is genuinely painful.",
        "Score": 76,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_fm2c5y",
        "Timestamp": "24/07/2025 13:20",
        "Text": "this is such an important point that gets lost in the consumer fraud angle. there are real ecological stakes here",
        "Score": 167,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_gn5d8z",
        "Timestamp": "25/07/2025 9:10",
        "Text": "me choosing banana boat reef friendly at the store: i am a responsible eco-conscious consumer\nalso me: inadvertently bringing chemicals harmful to coral to the reef\nthanks edgewell very cool",
        "Score": 423,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ho8e1a",
        "Timestamp": "25/07/2025 11:35",
        "Text": "The ACCC action is significant because Australia has relatively strong consumer protection law and the ACCC has a decent track record of actually winning these cases. Worth watching to see what penalty they seek - it could influence how other companies approach eco-claims.",
        "Score": 198,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_z4k2mb",
        "Timestamp": "25/07/2025 8:35",
        "Text": "I'd genuinely like to know what the internal conversations at Edgewell looked like when they decided to put 'reef friendly' on Banana Boat and Hawaiian Tropic products. Did anyone raise concerns? Did the marketing team just override the science team? Did they do any research at all or just look at what competitors were doing and copy it? These are the questions that would be interesting to see come out in Federal Court discovery. The ACCC case could surface a lot of internal documents about how these decisions get made. That's actually more valuable than the fine in terms of long-term change.",
        "Score": 57,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_ip1f6b",
        "Timestamp": "25/07/2025 12:50",
        "Text": "discovery documents in cases like this can be very revealing. fingers crossed the accc gets what it needs",
        "Score": 143,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_jq4g3c",
        "Timestamp": "26/07/2025 9:25",
        "Text": "Switched to Thinksport three summers ago and genuinely haven't looked back. yes its a bit more expensive but its actually what it says it is. novel concept for a sunscreen brand",
        "Score": 256,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_kr7h9d",
        "Timestamp": "26/07/2025 11:00",
        "Text": "hoping the accc case leads to australia adopting something like hawaii's ban on oxybenzone and octinoxate. that would be the actual meaningful regulatory outcome here",
        "Score": 187,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_a3n5vp",
        "Timestamp": "26/07/2025 8:50",
        "Text": "Let's talk about how this affects trust in eco labelling more broadly. The Edgewell greenwashing case - ACCC v Edgewell Personal Care, Federal Court, July 2025 - is going to make a lot of consumers rightfully sceptical of environmental claims on any product. And honestly, that scepticism is warranted. 'Natural', 'eco-friendly', 'sustainable', 'reef friendly', 'green' - most of these have no legal definition. They're aspiration words companies use to charge more money. The silver lining of this case is that it might push regulators to define these terms properly and punish companies that abuse them. Australia could actually lead on this.",
        "Score": 83,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_ls2i4e",
        "Timestamp": "26/07/2025 13:45",
        "Text": "yeah i'm now suspicious of every green label on everything and honestly thats probably the correct response. sad but correct",
        "Score": 289,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    }
]

In [6]:
# batch 2
batch_2 = [
    {
        "Post_id": "t3_b7n3kp",
        "Timestamp": "03/06/2022 10:14",
        "Text": "So I went down a rabbit hole last night reading about oxybenzone and coral reefs and I genuinely can't believe this stuff is still legal in most places. A 2016 study published in Archives of Environmental Contamination and Toxicology found that oxybenzone causes coral bleaching, DNA damage and deformities in larval coral at concentrations as low as 62 parts per trillion. That's the equivalent of one drop in 6.5 Olympic swimming pools. And yet Neutrogena, Coppertone, Banana Boat and dozens of other brands are still selling products with 4-6% oxybenzone concentration. The math on how much of this ends up in reef environments every summer is genuinely horrifying.",
        "Score": 84,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_cm4r9f",
        "Timestamp": "03/06/2022 11:30",
        "Text": "the 62 parts per trillion figure is what got me too. its not like you need a lot of it. one busy beach day puts way more than that into the water",
        "Score": 267,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_dn8s2g",
        "Timestamp": "03/06/2022 12:45",
        "Text": "Oxybenzone works as a UV filter by absorbing UV radiation and converting it to heat. The problem is coral polyps also absorb it and it acts as an endocrine disruptor, triggers viral infections in zooxanthellae (the algae coral depends on), and causes DNA damage. It basically turbocharges bleaching.",
        "Score": 312,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_c2m7wq",
        "Timestamp": "15/03/2023 9:05",
        "Text": "Hawaii banned oxybenzone and octinoxate in 2021 and Palau did it even earlier in 2020. These weren't just symbolic gestures - they were based on actual science. The Hawaiian legislature cited studies showing these chemicals were detectable in Hanauma Bay and other reef systems at concentrations above the damage threshold. The frustrating part is that the rest of the world looked at Hawaii and Palau and mostly... did nothing. The EU has been reviewing octocrylene and homosalate. The US FDA has been 'reviewing' oxybenzone since 2019. Meanwhile every beach in Australia, Southeast Asia and the Caribbean is getting dosed with this stuff every summer.",
        "Score": 76,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_eo3t5h",
        "Timestamp": "15/03/2023 10:20",
        "Text": "palau deserves more credit for being first on this. small island nation with the most to lose and they acted decisively while big countries dithered",
        "Score": 445,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_fp7u8i",
        "Timestamp": "15/03/2023 11:55",
        "Text": "The FDA 'reviewing' situation is maddening. They proposed rules in 2019 saying only zinc oxide and titanium dioxide were GRASE (generally recognized as safe and effective). Six years later oxybenzone products are still everywhere on US shelves.",
        "Score": 198,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_gq1v4j",
        "Timestamp": "16/03/2023 8:40",
        "Text": "switched to Badger Balm last year and honestly it goes on easier than i expected for a zinc oxide sunscreen. the sport version is great. def recommend for anyone wanting to ditch the chemical filters",
        "Score": 223,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_d5p9xr",
        "Timestamp": "22/07/2023 7:55",
        "Text": "Can we please stop the myth that 'mineral sunscreen' automatically means reef safe? Zinc oxide and titanium dioxide are safe BUT only in their non-nano form. Nano-sized zinc oxide particles (under 100nm) are small enough to be absorbed by coral tissue and can cause oxidative stress. Legitimate reef-safe brands like Stream2Sea and Raw Elements specifically use non-nano zinc oxide. When a product just says 'zinc oxide' without specifying non-nano, you don't actually know what you're getting. Always check the ingredient list AND look for brands that publish their particle size data.",
        "Score": 91,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_hr5w6k",
        "Timestamp": "22/07/2023 9:10",
        "Text": "this is such an underrated point. i thought i was doing the right thing by buying a zinc oxide sunscreen but never even thought to check if it was non-nano. back to reading labels i guess",
        "Score": 334,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_is9x3l",
        "Timestamp": "22/07/2023 10:35",
        "Text": "Stream2Sea actually tests their finished products on coral larvae and publishes the results on their website. That level of transparency is what every sunscreen company should be doing.",
        "Score": 276,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_jt2y7m",
        "Timestamp": "23/07/2023 8:20",
        "Text": "neutrogena ultra sheer has like 7.5% oxybenzone in it. its one of the bestselling sunscreens in the US. every summer millions of people slather this stuff on and jump in the ocean and we wonder why reefs are struggling",
        "Score": 389,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_e8q1ys",
        "Timestamp": "10/09/2023 11:00",
        "Text": "Breaking down the actual mechanism of oxybenzone coral damage for anyone who wants to understand the science rather than just trust the headlines. When oxybenzone is absorbed by coral, it photoactivates (gets energised by the same UV light it's supposed to block) and this triggers a cascade: it acts as an endocrine disruptor interfering with the coral's hormonal regulation, it causes viral dormancy to be broken (corals carry herpes-like viruses that oxybenzone can activate), and it induces genotoxicity meaning DNA strand breaks. Combined this means bleached coral, stunted larval development, and reduced reproductive success. The research here is solid - multiple independent labs have replicated these findings since the landmark 2016 Downs et al study.",
        "Score": 88,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_ku6z9n",
        "Timestamp": "10/09/2023 12:15",
        "Text": "really appreciate the detailed breakdown. do you have a link to the Downs et al study? want to read the primary source",
        "Score": 143,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_lv3a5o",
        "Timestamp": "10/09/2023 13:30",
        "Text": "the viral activation mechanism is the one that really disturbs me. coral already under thermal bleaching stress + chemical stress that activates latent viruses = death spiral",
        "Score": 256,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_mw7b2p",
        "Timestamp": "11/09/2023 9:00",
        "Text": "Coppertone sport still has oxybenzone prominently listed. Spent 20 minutes in the sunscreen aisle yesterday and could not find a single Coppertone product without it. Their 'pure & simple' line uses zinc but everything else... nope.",
        "Score": 178,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_f1r4zt",
        "Timestamp": "05/01/2024 8:30",
        "Text": "New year, new sunscreen routine. After properly researching this I've done a full ingredient audit of every sunscreen in my house. Results: 4 out of 6 contained oxybenzone, 2 contained octinoxate, 1 had both. All going in the bin. Replacing with Thinksport SPF 50 for daily use and Raw Elements for beach days. Yes the raw elements tin is a bit annoying to apply but it lasts forever and I know exactly what I'm putting on my skin and in the ocean. The white cast from non-nano zinc is worth it for me. Also both those brands are cheaper per use than the fancy Neutrogena stuff I was buying before so honestly it's a win.",
        "Score": 62,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_nx1c8q",
        "Timestamp": "05/01/2024 9:45",
        "Text": "thinksport is so underrated. does the job, clean ingredients, no fuss. been using it for 2 years",
        "Score": 312,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_oy4d6r",
        "Timestamp": "05/01/2024 11:00",
        "Text": "The white cast problem is real for deeper skin tones though. Non-nano zinc oxide leaves a significant white film that doesn't blend on darker skin. This is a genuine access and equity issue with reef-safe sunscreen - the safe alternatives work better for lighter skin tones.",
        "Score": 234,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_pz8e3s",
        "Timestamp": "06/01/2024 8:15",
        "Text": "iron oxide tinted zinc sunscreens help a lot with the white cast on darker skin. Badger Balm now has a tinted version. Still not perfect but way better than standard white zinc",
        "Score": 189,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_g4s7au",
        "Timestamp": "18/04/2024 10:20",
        "Text": "The octinoxate situation is slightly different from oxybenzone and I don't see people talk about this enough. While both are banned in Hawaii, the research on octinoxate coral toxicity is actually less robust than oxybenzone. The main concerns with octinoxate are: it's a strong endocrine disruptor (possibly more so than oxybenzone in humans), it photodegrades quickly so provides unreliable sun protection, and it has shown toxicity to marine organisms including coral in lab settings. But some researchers argue the field concentrations found near reefs are below damage thresholds. Point being: the science on octinoxate is still developing and the precautionary principle is being applied by Hawaii and Palau rather than definitive proof at the same level as oxybenzone.",
        "Score": 71,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_qa2f9v",
        "Timestamp": "18/04/2024 11:35",
        "Text": "appreciate the nuance here. too much reef safe discourse is all or nothing. understanding that the evidence is stronger for oxybenzone than octinoxate is useful",
        "Score": 167,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_rb5g4w",
        "Timestamp": "18/04/2024 13:00",
        "Text": "regardless of whether octinoxate is definitively proven at field concentrations - the precautionary principle absolutely applies here. we're talking about already stressed reef ecosystems. why add any unnecessary chemical risk?",
        "Score": 298,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_sc9h7x",
        "Timestamp": "19/04/2024 8:50",
        "Text": "banana boat has oxybenzone in most of their lineup and then puts 'reef friendly' on some products. the cognitive dissonance of the brand is wild. like do they understand what they're selling?",
        "Score": 423,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_h6t2bv",
        "Timestamp": "12/07/2024 9:15",
        "Text": "Quick guide to reading sunscreen labels for reef safety. Active ingredients to AVOID: oxybenzone (benzophenone-3), octinoxate (ethylhexyl methoxycinnamate), octocrylene, homosalate, octisalate, avobenzone. Active ingredients that are SAFE: zinc oxide (non-nano preferred), titanium dioxide (non-nano preferred). The thing is, these chemicals all have multiple names on labels. Oxybenzone is also listed as benzophenone-3 or BP-3. Octinoxate is also ethylhexyl methoxycinnamate or OMC. Companies sometimes use the less recognisable name specifically because fewer people know to avoid it.",
        "Score": 95,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_td3i5y",
        "Timestamp": "12/07/2024 10:30",
        "Text": "the alternate names thing is so manipulative. ethylhexyl methoxycinnamate sounds way more like a natural plant extract than octinoxate does",
        "Score": 356,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ue7j2z",
        "Timestamp": "12/07/2024 12:05",
        "Text": "saving this comment forever, been so confused by sunscreen labels. benzophenone-3 is oxybenzone??? i would never have known",
        "Score": 278,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_vf1k9a",
        "Timestamp": "13/07/2024 8:35",
        "Text": "Also worth knowing: avobenzone (butyl methoxydibenzoylmethane) is a chemical filter that is generally considered safer for reefs than oxybenzone/octinoxate - the data on reef toxicity is much weaker. It's not ideal but it's not in the same category. Don't conflate all chemical filters.",
        "Score": 213,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_i9u5cw",
        "Timestamp": "28/09/2024 11:45",
        "Text": "I took my family snorkelling on the Great Barrier Reef last month and the tour operator provided reef-safe sunscreen and made everyone use it. No personal sunscreen allowed in the water. I thought this was strict at the time but then I looked into the research and honestly they should be even stricter. A 2023 paper in Marine Pollution Bulletin found oxybenzone concentrations exceeding damage thresholds at popular snorkelling sites near Cairns during peak tourist season. The tourism industry is slowly getting it but it needs to be consistent and enforced not optional.",
        "Score": 79,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_wg4l3b",
        "Timestamp": "28/09/2024 13:00",
        "Text": "more tour operators should do this. its not that hard. you provide the sunscreen, problem solved. the barrier to switching is mostly inertia not technology",
        "Score": 245,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_xh8m6c",
        "Timestamp": "29/09/2024 8:20",
        "Text": "The Cairns data is alarming. There's a direct correlation between peak tourist season and elevated oxybenzone levels at popular snorkel sites. We're essentially adding chemical stress to reefs already under maximum thermal stress from warming waters.",
        "Score": 189,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_yi2n4d",
        "Timestamp": "29/09/2024 10:55",
        "Text": "neutrogena beach defense is literally one of the most popular sunscreens sold for beach and outdoor use and its loaded with homosalate and oxybenzone. the marketing is always beautiful beaches and ocean scenes. stunning lack of self awareness",
        "Score": 367,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_j2v8dx",
        "Timestamp": "14/02/2025 9:30",
        "Text": "I want to defend zinc oxide for a second because I see a lot of people complaining about the cosmetic downsides. Yes it leaves a white cast. Yes older formulations were thick and hard to apply. But modern non-nano zinc oxide formulations have improved massively. Brands like Thinksport and Stream2Sea have very workable textures now. Badger Balm's daily SPF 30 is basically invisible if you apply it properly. The technology has genuinely gotten better. The 'zinc is unusable' argument was more true in 2015 than it is in 2025. And frankly, if the choice is white cast vs contributing to coral bleaching, I'll take the white cast.",
        "Score": 67,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_zj5o7e",
        "Timestamp": "14/02/2025 10:45",
        "Text": "this. the formulations have gotten so much better. badger balm daily is legitimately cosmetically elegant now. the complaints are mostly from people who haven't tried recent versions",
        "Score": 234,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ak9p2f",
        "Timestamp": "14/02/2025 12:20",
        "Text": "still a white cast on deep brown skin no matter what formulation. acknowledge this instead of dismissing the concern. it's a real barrier for a lot of people",
        "Score": 312,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_bl3q8g",
        "Timestamp": "15/02/2025 8:40",
        "Text": "The white cast criticism is valid AND the reef toxicity problem is also real. Both can be true. The solution is better formulation technology not dismissing the concern. Cosmetically elegant reef-safe options for all skin tones is a genuine R&D gap.",
        "Score": 278,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_k5w1ey",
        "Timestamp": "03/04/2025 10:05",
        "Text": "Story time: I work as a marine biologist and every summer I get asked by well-meaning tourists what sunscreen to use. For years I gave nuanced answers about ingredients. Last year I started just telling people: if it has more than two active ingredients, put it back. If neither active ingredient is zinc oxide or titanium dioxide, put it back. This simple heuristic gets people 95% of the way there without needing a chemistry degree. The sunscreen industry has made this unnecessarily complicated. Mineral-only = generally safe. Mixed chemical + mineral or chemical-only = likely problematic for marine environments.",
        "Score": 88,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_cm7r5h",
        "Timestamp": "03/04/2025 11:20",
        "Text": "this heuristic is genuinely useful. more than two active ingredients = chemical filter present. putting this on my notes app",
        "Score": 389,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_dn1s9i",
        "Timestamp": "03/04/2025 13:45",
        "Text": "A marine biologist endorsing Thinksport and Stream2Sea is the only product rec I will ever need. bookmarking this thread",
        "Score": 267,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_eo4t3j",
        "Timestamp": "04/04/2025 8:55",
        "Text": "the 'more than two active ingredients' trick is genius. just tried it in the supermarket. every single chemical sunscreen had 4+ actives. every mineral one had 1 or 2. works perfectly",
        "Score": 445,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_l8x4fz",
        "Timestamp": "19/05/2025 9:40",
        "Text": "Something I've never seen discussed: oxybenzone is also absorbed through human skin and detected in blood, breast milk and urine. So it's not just a marine ecology issue - it's a human health issue. The FDA's proposed GRASE rules from 2019 raised this specifically, noting insufficient data on whether oxybenzone is safe for human use at the concentrations used in sunscreens. You're literally absorbing a known endocrine disruptor every time you use it. The reef argument gets more traction but honestly the human health argument should be equally compelling to people who don't live near reefs.",
        "Score": 74,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_fp2u6k",
        "Timestamp": "19/05/2025 10:55",
        "Text": "detected in breast milk is the one that made me actually change my habits. not just for coral anymore - for my kid",
        "Score": 423,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_gq6v1l",
        "Timestamp": "19/05/2025 12:30",
        "Text": "Important caveat: the presence in blood doesn't automatically mean it causes harm. The FDA said more research is needed - that's different from saying it's dangerous. But 'more research needed' + endocrine disruption potential + reef toxicity proven = precautionary principle applies strongly here",
        "Score": 198,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_hr9w4m",
        "Timestamp": "20/05/2025 9:10",
        "Text": "coppertone water babies has oxybenzone in it. the product specifically marketed for babies and children to use at the beach contains an ingredient that is a suspected endocrine disruptor and marine toxin. make it make sense",
        "Score": 478,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_m1y7ga",
        "Timestamp": "08/06/2025 8:25",
        "Text": "Genuine question: why is there such a gap between the scientific consensus on oxybenzone harm and what's actually on shop shelves? I've read the Downs et al studies, I've read the Hawaii legislative record, I've read the FDA GRASE proposal. The science seems clear. And yet I can walk into any pharmacy and buy Neutrogena with 7% oxybenzone right now. What is the actual barrier to change? Is it lobbying by chemical companies? Regulatory capture? Consumer inertia? I genuinely want to understand why the market hasn't moved faster on this even in the absence of regulation.",
        "Score": 82,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_is3x8n",
        "Timestamp": "08/06/2025 9:40",
        "Text": "the answer is mostly: it's cheap, it works well as a UV filter, and changing formulations costs money. mineral sunscreens are harder to formulate elegantly and more expensive. no financial incentive to change unless regulation forces it",
        "Score": 234,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_jt7y2o",
        "Timestamp": "08/06/2025 11:15",
        "Text": "Sunscreen Chemical Coalition absolutely lobbied against stronger FDA restrictions. This is documented. The industry argued the existing safety record was sufficient and new testing requirements were unnecessary. Classic regulatory delay playbook.",
        "Score": 312,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ku1z5p",
        "Timestamp": "09/06/2025 8:30",
        "Text": "Raw Elements is worth every cent. tiny bit of product goes a long way, genuinely water resistant, and i feel good about using it near coral. found it years ago and never went back to chemical sunscreen",
        "Score": 189,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_n4z3hb",
        "Timestamp": "24/06/2025 9:55",
        "Text": "Just got back from snorkelling in Exmouth at Ningaloo Reef. The dive company gave us all Stream2Sea sunscreen and explained why chemical sunscreens aren't allowed in their operations. Had a long chat with the guide about the oxybenzone research. She said they've noticed a visible difference in coral health in the areas where they enforce the rule versus popular spots where people use whatever sunscreen they brought. Obviously that's anecdotal not a controlled study, but it aligned perfectly with the lab research. Small conscious choices at scale really do matter.",
        "Score": 68,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_lv5a9q",
        "Timestamp": "24/06/2025 11:10",
        "Text": "Ningaloo is one of the few places left with genuinely pristine coral and they take this seriously. good to hear the operators are enforcing it",
        "Score": 267,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_mw9b4r",
        "Timestamp": "24/06/2025 12:45",
        "Text": "stream2sea was specifically developed for use near coral reefs - the founder is a marine biologist and they test on live coral larvae. that origin story matters. this wasn't just a brand that reformulated to chase a trend, it was built for this purpose from day one",
        "Score": 356,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_nx3c7s",
        "Timestamp": "25/06/2025 9:20",
        "Text": "honest question - if zinc oxide and titanium dioxide are so clearly safer why do brands like Neutrogena and Banana Boat keep using chemical filters at all? cost difference cant be THAT significant for a multinational right",
        "Score": 143,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_oy6d2t",
        "Timestamp": "25/06/2025 10:55",
        "Text": "It actually is a significant cost difference at scale plus formulation difficulty. Chemical filters are also easier to make water resistant and cosmetically elegant. The consumer preference for lightweight non-whitening sunscreen has historically favoured chemical formulations. Changing would require reformulation investment and accepting some customer complaints about texture. Short-term cost vs long-term responsibility basically.",
        "Score": 224,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    }
]

In [7]:
# batch 3
batch_3 = [
    {
        "Post_id": "t3_n7k2qp",
        "Timestamp": "11/05/2022 9:18",
        "Text": "Ok I finally made the switch to reef safe sunscreen after years of procrastinating and I need to talk about Stream2Sea. I was fully prepared to hate it because I've tried mineral sunscreens before that left me looking like a mime. This is not that. It blends in reasonably well, actually stays on in the water, and the SPF 30 formula for face doesn't break me out. I've been using it for two beach trips now. Yes it's more expensive than grabbing Banana Boat off the shelf but I feel genuinely good about what I'm putting in the ocean. Also their entire packaging is biodegradable including the tube. That's the cherry on top.",
        "Score": 86,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_ao3r4f",
        "Timestamp": "11/05/2022 10:35",
        "Text": "stream2sea converted me too. the biodegradable packaging was the thing that pushed me over the edge. what's the point of a reef safe product that comes in plastic that ends up in the ocean anyway",
        "Score": 289,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_bp6s7g",
        "Timestamp": "11/05/2022 11:50",
        "Text": "The Stream2Sea founder is a marine toxicologist which is why the formulation is genuinely rigorous. They test on coral larvae and sea urchin embryos. It's not just 'no bad ingredients' - they actively verify the finished product is non-toxic to marine life.",
        "Score": 334,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_o4m5wr",
        "Timestamp": "22/08/2022 8:45",
        "Text": "Can anyone help me find a reef safe sunscreen that actually works for sport/outdoor activities? I've been using Neutrogena sport for years but after reading about oxybenzone I want to switch. The problem is every mineral sunscreen I've tried just slides off my face the minute I start sweating. I'm a runner, I need something that STAYS. Budget is flexible I just need it to actually function.",
        "Score": 73,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_cq9t2h",
        "Timestamp": "22/08/2022 9:55",
        "Text": "Thinksport SPF 50 is your answer. specifically formulated for sport, very water and sweat resistant, passes EWG standards. been using it for trail running for 3 years. does not move.",
        "Score": 412,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_dr2u8i",
        "Timestamp": "22/08/2022 11:20",
        "Text": "seconding Thinksport. also Raw Elements Sport is excellent for outdoor activities. slightly thicker than Thinksport but bulletproof once it's on. I do open water swimming and it's the only mineral sunscreen that's ever stayed on for me",
        "Score": 267,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_es5v4j",
        "Timestamp": "23/08/2022 8:30",
        "Text": "For sport specifically I'd also mention Badger Balm Sport SPF 35. It's thicker than Thinksport but the water resistance is excellent and it's EWG verified. The tin is a bit awkward for on-the-go but I keep it in my kit bag.",
        "Score": 178,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_p8n1xs",
        "Timestamp": "14/01/2023 10:05",
        "Text": "Long post incoming because I want to properly explain why I've been recommending Raw Elements to everyone I know for the past year. I made the switch after researching oxybenzone for a trip to the Great Barrier Reef. What sold me on Raw Elements specifically is the tin packaging - no plastic at all, infinitely recyclable, and the tin is so sturdy it's survived in my beach bag through absolute carnage. The formula is non-nano zinc oxide only, no titanium dioxide, no chemical filters. It's EWG verified. The SPF 30 tin lasts me a surprisingly long time because a little goes far. Does it leave a slight white cast on my arms? Yes. Do I care? No. The reef is more important than looking slightly chalky for an afternoon.",
        "Score": 91,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_ft8w6k",
        "Timestamp": "14/01/2023 11:20",
        "Text": "the tin packaging is what got me too. genuinely no plastic in the whole product. rare",
        "Score": 356,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_gu2x3l",
        "Timestamp": "14/01/2023 12:45",
        "Text": "Raw Elements is EWG verified which means it's gone through independent third party testing - that's not just their own claim. The EWG verification program is actually rigorous compared to brands that self certify.",
        "Score": 234,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_hv5y7m",
        "Timestamp": "15/01/2023 9:10",
        "Text": "I switched from Hawaiian Tropic to Raw Elements two years ago and genuinely cannot imagine going back. not even slightly tempted by the price difference anymore",
        "Score": 189,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_q1p4yt",
        "Timestamp": "05/04/2023 8:35",
        "Text": "Let me explain what EWG verification actually means because I see it on Thinksport and Raw Elements and Badger Balm and people often don't know what they're paying for. The Environmental Working Group runs a verification program where companies submit full ingredient lists, product formulations, and safety data for independent review. EWG checks for chemicals of concern, concentration levels, and whether claims are substantiated. For sunscreen specifically, they check that the active ingredient is zinc oxide or titanium dioxide at appropriate concentrations, and that inactive ingredients don't include anything problematic. It's not perfect and some good brands don't bother getting it, but EWG verified is a meaningful signal that a product has been externally scrutinised.",
        "Score": 84,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_iw9z2n",
        "Timestamp": "05/04/2023 9:50",
        "Text": "this should be pinned in every sunscreen thread. so many people don't understand what EWG verification actually involves",
        "Score": 278,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_jx3a8o",
        "Timestamp": "05/04/2023 11:15",
        "Text": "worth noting EWG has a free sunscreen database at ewg.org/sunscreen where you can look up any brand. I use it before every purchase. Neutrogena and Coppertone score terribly, Thinksport and Raw Elements score in the top tier",
        "Score": 312,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ky6b5p",
        "Timestamp": "06/04/2023 8:20",
        "Text": "I ran every sunscreen in my house through the EWG database last year and threw out everything that scored above 3. Replaced with Badger Balm and haven't looked back. Score 1 products across the board now.",
        "Score": 223,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_r7q6zu",
        "Timestamp": "18/07/2023 9:25",
        "Text": "Badger Balm appreciation post. I've been using it for three years now and it's genuinely one of those products where I can't believe it took me so long to find it. The daily SPF 30 is the most cosmetically elegant zinc oxide sunscreen I've ever used - it's tinted so no white cast, it doubles as a moisturiser, and it smells faintly of chamomile. The sport version is thicker but basically welded to your skin once applied. Both are certified organic, EWG verified, cruelty free and packaged in either recyclable tubes or recycled metal tins. The price is comparable to what I was spending on Banana Boat honestly.",
        "Score": 77,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_lz1c9q",
        "Timestamp": "18/07/2023 10:40",
        "Text": "the tinted daily one is a sleeper hit. i use it as my everyday face spf. works under makeup and doesn't pill. would never go back to a chemical sunscreen",
        "Score": 389,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ma4d3r",
        "Timestamp": "18/07/2023 12:05",
        "Text": "Badger Balm is family owned and B Corp certified too. I appreciate supporting a company with actual values rather than a multinational that puts reef friendly on products containing oxybenzone",
        "Score": 245,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_nb8e7s",
        "Timestamp": "19/07/2023 9:00",
        "Text": "fair criticism: Badger Balm is not cheap. the sport tin is like $18-20 for 2.9oz. Thinksport gives you more product for less money if budget is a concern. both are good, Thinksport wins on value",
        "Score": 167,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_s2r5av",
        "Timestamp": "03/10/2023 8:50",
        "Text": "Hot take that I'll defend: Sun Bum is a perfectly good gateway reef safe sunscreen. Yes it's not as rigorously tested as Stream2Sea. Yes some formulations have avobenzone which is technically a chemical filter. But the SPF 50 mineral version is non-nano zinc oxide only, it's affordable at Target, and it gets people off oxybenzone products. Perfect shouldn't be the enemy of good here. If someone is choosing between Neutrogena with 7% oxybenzone and Sun Bum mineral, Sun Bum is meaningfully better. We should celebrate that reef-safer options are now available at mainstream retailers instead of being snobby about it.",
        "Score": 82,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_oc1f2t",
        "Timestamp": "03/10/2023 10:05",
        "Text": "strong agree. accessibility matters. not everyone can order Stream2Sea online and wait for delivery. having mineral options at the petrol station or pharmacy is genuinely progress",
        "Score": 334,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_pd5g8u",
        "Timestamp": "03/10/2023 11:30",
        "Text": "The Sun Bum branding is also just... cooler than most mineral sunscreens which tend to look like they're aimed at senior citizens. That matters for getting younger people to switch.",
        "Score": 267,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_qe9h4v",
        "Timestamp": "04/10/2023 8:15",
        "Text": "counterpoint: Sun Bum's non-mineral range still contains chemical filters including homosalate and it markets the whole brand as beach/reef lifestyle. the mixed messaging is annoying. but yeah the mineral line specifically is fine",
        "Score": 198,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_t6s8bw",
        "Timestamp": "15/01/2024 9:35",
        "Text": "switching to reef safe sunscreen is one of those things that felt like a big deal before I did it and then turned out to be a complete non-event. My switch journey: researched for about a week, decided on Thinksport SPF 50 as my main sunscreen because it had good reviews for water resistance. Ordered online. Used it. It was fine. Literally just fine. I'm not saying this to be dismissive - I mean it works well, I like using it, the texture is good. But I built it up in my head as this huge inconvenient sacrifice and it was just... buying a different sunscreen. If you've been putting it off, just do it. The first step is picking one of the known good brands and ordering it.",
        "Score": 71,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_rf3i5x",
        "Timestamp": "15/01/2024 10:50",
        "Text": "this is the most helpful framing. it really is just buying a different sunscreen. i overthought it too",
        "Score": 423,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_sg7j2y",
        "Timestamp": "15/01/2024 12:15",
        "Text": "Thinksport was my gateway too. Once I realised reef safe sunscreen just works like normal sunscreen I started switching other products. Kind of a gateway to looking more carefully at everything.",
        "Score": 189,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_th1k8z",
        "Timestamp": "16/01/2024 8:40",
        "Text": "the price thing is real though. Thinksport at ~$15 for 3oz vs Banana Boat at ~$8 for 8oz. per oz the cost is significant especially for families with multiple kids. wish the economics were better",
        "Score": 234,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_u9t1cx",
        "Timestamp": "28/03/2024 10:20",
        "Text": "I want to give Stream2Sea a proper write up because it's criminally underrated. Founded by a marine toxicologist who literally studies how sunscreen chemicals harm marine ecosystems. They test every product formulation on coral larvae, sea urchin embryos, and baby brine shrimp before it goes to market. If it kills any of these test organisms, it doesn't get released. That's a genuinely rigorous standard that no mainstream brand comes close to. Their packaging is biodegradable - the tube breaks down in marine environments which is obviously ideal. SPF 20 and 30 face and body formulations available. The texture is better than I expected for a marine-tested product. Not the cheapest but you're paying for actual science.",
        "Score": 89,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_ui4l3a",
        "Timestamp": "28/03/2024 11:35",
        "Text": "the biodegradable tube is honestly more impressive than any ingredient list. packaging is the thing nobody talks about enough. all your reef-safe ingredients are meaningless if the microplastic tube ends up in the ocean",
        "Score": 356,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_vj8m7b",
        "Timestamp": "28/03/2024 13:00",
        "Text": "Tested on coral larvae is such a high bar. Not theoretical safety based on ingredient lists but actual empirical testing. This is what all marine environment claims should require.",
        "Score": 278,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_wk2n4c",
        "Timestamp": "29/03/2024 8:55",
        "Text": "I've been recommending Stream2Sea to every dive buddy for two years. Once you explain that the founder is literally a marine toxicologist who created it specifically to stop harming coral, people get it immediately. The story sells itself.",
        "Score": 212,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_v3u5dy",
        "Timestamp": "12/06/2024 9:10",
        "Text": "Comparing Thinksport and Raw Elements side by side because I've used both extensively. Thinksport SPF 50: lighter texture, easier to apply, better for everyday face use, slightly less white cast. Great for kids because it goes on fast. Price is better per oz. Raw Elements SPF 30: thicker, takes more effort to blend, MUCH better water resistance for swimming and surfing, packaging is infinitely better (metal tin). For sitting on the beach Thinksport. For actually being in the water, Raw Elements. For zero waste packaging, Raw Elements by a mile. Both are EWG verified and use non-nano zinc oxide only. You genuinely can't go wrong with either.",
        "Score": 78,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_xl6o1d",
        "Timestamp": "12/06/2024 10:25",
        "Text": "this is the comparison I needed. been using Thinksport but do a lot of ocean swimming so switching to Raw Elements for beach days based on this. thanks",
        "Score": 289,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ym9p5e",
        "Timestamp": "12/06/2024 12:00",
        "Text": "raw elements tin has survived in my beach bag for two summers. indestructible. compared to every plastic tube I've ever owned that split and emptied itself into my bag",
        "Score": 367,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_zn3q9f",
        "Timestamp": "13/06/2024 8:30",
        "Text": "minor gripe: the raw elements tin is annoying to get into when your hands are wet and sandy. use the back of your fingers to scoop. once you get the technique it's fine but the learning curve is real lol",
        "Score": 156,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_w7v2ez",
        "Timestamp": "09/08/2024 8:40",
        "Text": "I convinced my whole household to switch to reef safe sunscreen and here's how. Step one: I stopped buying Neutrogena and Banana Boat without announcing it. Step two: replaced with Thinksport for kids and Badger Balm daily for adults. Step three: waited to see if anyone noticed. Result: nobody said anything for two months. My husband eventually said his face felt better since 'the sunscreen change'. My kids use whatever I hand them. The switch cost about $15 more per year total. The thing I thought would be a big fight turned out to be completely invisible. Sometimes you just have to do the thing instead of asking for permission.",
        "Score": 94,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_ao4r7g",
        "Timestamp": "09/08/2024 9:55",
        "Text": "lmaooo the stealth sustainability switch. this is the way honestly. family members who resist change often don't notice when you just quietly do it",
        "Score": 445,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_bp8s3h",
        "Timestamp": "09/08/2024 11:20",
        "Text": "this works for lots of things. switched cleaning products, toilet paper, dish soap. nobody noticed any of it. the 'my family won't go for it' excuse is often just inertia talking",
        "Score": 312,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_cq1t6i",
        "Timestamp": "10/08/2024 8:15",
        "Text": "Thinksport for kids is specifically worth calling out. tested and free of the nasty stuff, decent water resistance, doesn't sting eyes as badly as chemical sunscreens. genuinely good kids product",
        "Score": 223,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_x5w8fa",
        "Timestamp": "25/10/2024 10:30",
        "Text": "Can we have a thread about reef safe sunscreen AND plastic free packaging? Because most of the reef safe conversation is about ingredients which is great but the plastic tube still ends up in landfill or ocean. Here's my breakdown: Raw Elements wins on packaging (metal tin, no plastic). Stream2Sea wins on biodegradable packaging (the tube itself breaks down). Badger Balm has recyclable packaging and uses post-consumer recycled materials. Thinksport is the weakest on packaging - standard plastic tube but recyclable. If you're trying to minimise total ocean impact ingredients AND packaging both matter.",
        "Score": 66,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_dr5u2j",
        "Timestamp": "25/10/2024 11:45",
        "Text": "the packaging angle is underappreciated. even if your sunscreen has safe ingredients the plastic tube is going to outlast coral reefs as we currently know them",
        "Score": 289,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_es9v6k",
        "Timestamp": "25/10/2024 13:10",
        "Text": "Stream2Sea biodegradable tube is the gold standard imo. you could literally drop it in the ocean (don't, but you could) and it would break down. no other sunscreen brand can say that",
        "Score": 378,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ft2w9l",
        "Timestamp": "26/10/2024 8:40",
        "Text": "for the record Badger Balm uses PCR plastic and their cardboard outer packaging is FSC certified. not zero waste but better than most. also they have a refillable program at some retailers",
        "Score": 167,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_y2x1gb",
        "Timestamp": "14/02/2025 9:00",
        "Text": "Six months update on my reef safe sunscreen journey. I started with Thinksport SPF 50 in August, added Badger Balm daily tinted for face in October, tried Stream2Sea on a diving trip in January. Current verdict: Thinksport is my everyday body sunscreen and probably will be forever. Badger Balm tinted is my face sunscreen. Stream2Sea is my ocean activities sunscreen. Total monthly cost is basically what I was spending on Neutrogena and Hawaiian Tropic before, maybe $2-3 more. Zero regrets. The white cast thing people warn you about is genuinely not a big deal in practice. You get used to it in about a week.",
        "Score": 81,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_gu6z4m",
        "Timestamp": "14/02/2025 10:15",
        "Text": "great update! the Badger Balm tinted for face + Thinksport for body combination is exactly what I do. perfect system",
        "Score": 234,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_hv1a8n",
        "Timestamp": "14/02/2025 11:40",
        "Text": "the white cast adjustment period is real - first few times I used non-nano zinc I was self-conscious. now i genuinely don't notice or care. perspective shifts",
        "Score": 189,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_iw4b3o",
        "Timestamp": "15/02/2025 8:25",
        "Text": "I appreciate these long term updates. it's one thing to say 'I tried it once and it was fine' and another to say 'I've been using it for 6 months and this is my considered opinion'. the latter is actually useful",
        "Score": 312,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_z6y4hc",
        "Timestamp": "08/04/2025 9:45",
        "Text": "Genuine frustration: why are these brands so hard to find in physical stores? I can get Thinksport online easy but if I forget sunscreen on a road trip and stop at a petrol station I'm back to choosing between Banana Boat and nothing. Stream2Sea is practically invisible at retail. Raw Elements is online only in most markets. Badger Balm is in some health food stores but not everywhere. The brands doing the right thing are penalised with limited distribution because they can't afford supermarket shelf fees. This is a structural problem that consumer choice alone can't fix. We need retailers to stock this stuff.",
        "Score": 75,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_jx8c6p",
        "Timestamp": "08/04/2025 11:00",
        "Text": "this is my main frustration too. i now keep a backup tin of raw elements in my car for exactly this reason. once you commit to not buying chemical sunscreen you realise how few options there are when you're caught without",
        "Score": 267,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ky3d2q",
        "Timestamp": "08/04/2025 12:25",
        "Text": "Sun Bum mineral is at Target and Woolworths which helps for accessibility. Not the deepest reef-safe brand but better than the alternative in an emergency situation",
        "Score": 223,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_lz7e9r",
        "Timestamp": "09/04/2025 8:35",
        "Text": "I emailed my local pharmacy chain asking them to stock Thinksport and Badger Balm. They said they get this request occasionally and are reviewing their range. If enough people ask retailers directly it does have some effect. worth doing",
        "Score": 156,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ma1f4s",
        "Timestamp": "09/04/2025 10:00",
        "Text": "For the record Thinksport is now on Amazon Prime which at least solves the 'I need it quickly' problem even if it doesn't solve the 'I forgot it at the beach' problem. still carry an emergency tin in the car though",
        "Score": 178,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    }
]

In [8]:
# batch 4
batch_4 = [
    {
        "Post_id": "t3_c4k8np",
        "Timestamp": "07/03/2022 10:22",
        "Text": "Spent 25 minutes in the sunscreen aisle at Chemist Warehouse yesterday trying to find something genuinely reef safe. Every single product I picked up said either 'reef friendly', 'ocean safe', 'marine safe', or 'eco formula'. Every single one. Then I turned them over and read the ingredients. Oxybenzone. Octinoxate. Homosalate. Octocrylene. On products with turtles and coral printed on the front. I left without buying anything because I genuinely could not tell which claims, if any, were true. This is not a consumer education problem. This is a labelling fraud problem. If I can't trust what's on the front of a bottle then the label is worse than useless.",
        "Score": 92,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_dm2s6f",
        "Timestamp": "07/03/2022 11:40",
        "Text": "the turtle on the bottle while containing oxybenzone is genuinely one of the most cynical things in consumer products. the symbol of what they're destroying used to sell the product destroying it",
        "Score": 467,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_en6t3g",
        "Timestamp": "07/03/2022 13:05",
        "Text": "There is literally no legal definition for 'reef safe' or 'reef friendly' in Australia, the US, or most of the EU. Any brand can put it on any product. It means exactly as much as writing 'ocean vibes' on the label.",
        "Score": 312,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_fo9u7h",
        "Timestamp": "08/03/2022 8:30",
        "Text": "I called the customer service line for Banana Boat once to ask what their 'reef friendly' claim was based on. After being on hold for 12 minutes they told me it meant the product 'met their internal environmental standards'. I asked what those standards were. They said they couldn't share that information. I hung up.",
        "Score": 489,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_d7m1qr",
        "Timestamp": "19/06/2022 9:15",
        "Text": "Hot take: 'reef friendly' on sunscreen is more misleading than no environmental claim at all. When a product makes no sustainability claims, at least you know to do your own research. When it says 'reef friendly' most consumers reasonably assume someone has checked that it's actually reef friendly. They haven't. Nobody has. There's no body that does this. There's no standard it has to meet. The claim is pure marketing designed to extract money from environmentally conscious consumers who are doing their best. And we have to pay a premium for the privilege of being lied to.",
        "Score": 84,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_gp3v4i",
        "Timestamp": "19/06/2022 10:30",
        "Text": "the premium pricing is what kills me. you're paying extra specifically because you care about the reef. and that caring is being monetised by brands that are actively harming the reef. evil genius tbh",
        "Score": 378,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_hq7w2j",
        "Timestamp": "19/06/2022 11:55",
        "Text": "I've started calling this the eco-conscience tax. Companies charge you more for the FEELING of doing something good without actually delivering the thing. Happens with sunscreen, cleaning products, food packaging, basically everything.",
        "Score": 234,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ir1x8k",
        "Timestamp": "20/06/2022 8:45",
        "Text": "Hawaiian Tropic literally has the word TROPIC in the name. Their whole brand is built on imagery of pristine tropical marine environments. And they sell products containing oxybenzone which damages those exact environments. The audacity is breathtaking.",
        "Score": 423,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_e2n4xs",
        "Timestamp": "04/09/2022 10:05",
        "Text": "I need to vent about something that happened at a beach shop in Cairns. I specifically asked the staff member which of their sunscreens were reef safe for snorkelling. She pointed me to the 'reef friendly' range. I asked what made them reef friendly. She said they were specially formulated. I asked for ingredients. She didn't know. I looked myself and found oxybenzone in the second product she recommended. She genuinely believed she was helping me make a responsible choice. She wasn't wrong to trust the label. The label was wrong. The entire system failed at every level except the consumer who did the work to actually read the ingredient list.",
        "Score": 88,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_js4y5l",
        "Timestamp": "04/09/2022 11:20",
        "Text": "This exact thing happens constantly in tourist areas near reefs. Staff at shops near the GBR are telling tourists to buy 'reef friendly' labelled products in good faith, not knowing the label is meaningless. The systemic failure goes all the way down.",
        "Score": 267,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_kt8z9m",
        "Timestamp": "04/09/2022 13:40",
        "Text": "and the tourist who bought it goes home thinking they did the right thing. and comes back next year and does the same thing. the misinformation compounds",
        "Score": 345,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_lu2a3n",
        "Timestamp": "05/09/2022 8:55",
        "Text": "The retailers near tourist reef sites should bear some responsibility here too. If you operate a business near the Great Barrier Reef you should not be stocking products with oxybenzone regardless of what the label says. Do the research once, stock accordingly.",
        "Score": 198,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_f5o7yt",
        "Timestamp": "12/11/2022 9:30",
        "Text": "Can we talk about influencer greenwashing in the sunscreen space? I follow several sustainability influencers and at least three of them have done sponsored posts for sunscreens using 'reef safe' claims that, when I checked the ingredients, contained chemical filters that are demonstrably harmful to marine environments. They either didn't check the ingredients or checked and didn't care because of the sponsorship money. Either way they're spreading misinformation to hundreds of thousands of followers who trust them specifically for sustainable living advice. This feels worse than brand advertising because there's an implied personal endorsement of values.",
        "Score": 79,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_mv5b6o",
        "Timestamp": "12/11/2022 10:45",
        "Text": "sustainability influencer sponsored content is almost always compromised. if the brand is paying them, the brand controls the message. the cognitive dissonance required to post reef safe sponsored content without reading the label is wild",
        "Score": 356,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_nw9c4p",
        "Timestamp": "12/11/2022 12:10",
        "Text": "I've started commenting ingredient callouts on these posts. Gets me blocked pretty fast but at least some people in the comments see it before it gets removed.",
        "Score": 289,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ox3d8q",
        "Timestamp": "13/11/2022 8:20",
        "Text": "Neutrogena has a whole 'Pure and Free' baby range that markets itself heavily to eco-conscious parents. Checked the mineral baby sunscreen - ok that one's actually fine. Checked the 'Ultra Sheer Dry Touch' that's right next to it in the same branding section with the same green packaging - 7.5% oxybenzone. Same shelf, same visual design, one is fine, one isn't. Average parent grabs whichever is on sale.",
        "Score": 412,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_g8p2zu",
        "Timestamp": "08/02/2023 10:00",
        "Text": "I've been researching the regulatory landscape for sunscreen claims in different countries and the short version is: it's a mess everywhere but some places are less messy. Australia: no legal definition for reef safe/friendly, ACCC has taken some action on specific misleading claims (notably the Edgewell case in 2025). US: FDA regulates sunscreen as OTC drug but doesn't regulate environmental claims, which fall to FTC green guide guidelines that have no enforcement teeth. EU: moving toward stricter green claims regulation but not there yet. Hawaii and Palau: actual ingredient bans which are the most effective approach. Everyone else: chaos. The answer globally is ingredient bans, not label regulation, because labels can always be gamed.",
        "Score": 86,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_py6e3r",
        "Timestamp": "08/02/2023 11:15",
        "Text": "the FTC green guides are essentially toothless in practice. guidance that isn't enforced isn't really regulation. Hawaii's ingredient ban is the only model that actually works",
        "Score": 223,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_qz9f7s",
        "Timestamp": "08/02/2023 12:40",
        "Text": "Australia's ACCC at least has a track record of actually suing companies. The Edgewell Federal Court case is significant because it's not just a warning letter, it's proper enforcement. Other regulators should take note.",
        "Score": 178,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ra3g2t",
        "Timestamp": "09/02/2023 8:50",
        "Text": "ingredient bans vs label regulation is such a key distinction. Label regulation just means brands get creative with wording. If oxybenzone is banned full stop, there's nothing to greenwash.",
        "Score": 312,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_h1q5av",
        "Timestamp": "27/04/2023 9:20",
        "Text": "Coppertone 'Pure and Simple' range. Let me tell you about my experience with this. The packaging is minimalist white and green, the word 'pure' is in the name, there are no bold chemical names on the front. I bought it at the airport because I needed sunscreen and it looked clean. Got home, looked up the full ingredient list online because I forgot to check at the shop. Avobenzone, homosalate, octisalate, and octocrylene. Four chemical filters. 'Pure and Simple'. I'm not sure I've ever felt more manipulated by packaging design in my life. The entire aesthetic was engineered to read as minimal and clean while containing a cocktail of chemical UV filters.",
        "Score": 91,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_sb7h4u",
        "Timestamp": "27/04/2023 10:35",
        "Text": "'Pure and Simple' containing four chemical filters is genuinely impressive greenwashing. the name does ALL the heavy lifting so the ingredients don't have to",
        "Score": 445,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_tc1i9v",
        "Timestamp": "27/04/2023 12:00",
        "Text": "the minimalist white and green packaging is a whole design language that says 'clean and natural' with zero legal basis. brands spend millions making products LOOK safe so you don't check if they ARE",
        "Score": 367,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ud5j3w",
        "Timestamp": "28/04/2023 8:30",
        "Text": "I've started doing a rule at the pharmacy: if the packaging has any green colour, any ocean or nature imagery, or any eco-sounding words, I assume it's probably greenwashing until proven otherwise by the ingredient list. sad state of affairs",
        "Score": 234,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_i4r8bw",
        "Timestamp": "15/07/2023 10:10",
        "Text": "The funniest/most depressing thing I've seen recently: a major pharmacy chain had a whole display labelled 'Reef Safe Sunscreens' featuring Banana Boat 'reef friendly' prominently. The ACCC had already signalled concern about Banana Boat's reef friendly claims. The pharmacy either didn't know, didn't care, or calculated that the greenwashing display would outsell the liability. Every person who walked past that display and trusted it was failed by the retailer, the brand, and the regulator simultaneously. This is what regulatory gaps look like in practice.",
        "Score": 77,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_ve9k7x",
        "Timestamp": "15/07/2023 11:25",
        "Text": "pharmacy chains should be liable too honestly. if you curate a 'reef safe' display you're making an implicit claim about the products in it. that should come with some responsibility",
        "Score": 289,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_wf3l2y",
        "Timestamp": "15/07/2023 12:50",
        "Text": "Retailers absolutely know the labels are vague. They benefit from the consumer confusion because it drives sales. 'Reef Safe Display' is a footfall driver not an editorial judgement.",
        "Score": 198,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_xg7m5z",
        "Timestamp": "16/07/2023 8:40",
        "Text": "I photographed that exact kind of display at a Chemist Warehouse near the GBR and sent it to the ACCC. Got an automated acknowledgement. Doubt anything happened but felt marginally better",
        "Score": 156,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_j6s1cx",
        "Timestamp": "02/10/2023 9:45",
        "Text": "I want to explain what makes the 'reef friendly' label specifically infuriating compared to other greenwashing. Most greenwashing is vague - 'sustainable', 'eco-conscious', 'green'. These are weasel words that can't really be disproved. 'Reef friendly' is different because it's a specific empirical claim about a specific ecosystem. There is actual science on what harms reefs. The ingredients in many 'reef friendly' products are demonstrably, measurably harmful to coral at concentrations that occur in reef environments. It's not a matter of interpretation. The claim is factually false. That's not greenwashing in the usual sense - that's product mislabelling.",
        "Score": 95,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_yh4n8a",
        "Timestamp": "02/10/2023 11:00",
        "Text": "the distinction between vague aspirational greenwashing and specific false claims is really important and doesn't get made often enough. reef friendly is in the second category",
        "Score": 278,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_zi8o4b",
        "Timestamp": "02/10/2023 12:25",
        "Text": "This is exactly the legal theory the ACCC is using in the Edgewell case. Misleading conduct under Australian Consumer Law requires that a specific false impression be created in the consumer's mind. 'Reef friendly' creates the specific impression that the product doesn't harm reefs. That impression is false.",
        "Score": 334,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_aj2p7c",
        "Timestamp": "03/10/2023 8:15",
        "Text": "meanwhile brands will argue in court that 'reef friendly' is merely aspirational marketing language that no reasonable consumer would take as a literal scientific claim. watch for that argument. it's coming.",
        "Score": 212,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_k9t4dy",
        "Timestamp": "18/01/2024 10:30",
        "Text": "My experience trying to buy sunscreen for a trip to the Whitsundays. Searched 'reef safe sunscreen Australia' - first three results were Banana Boat reef friendly, Hawaiian Tropic reef friendly, and a Neutrogena product with a 'reef aware' label I'd never seen before. Clicked on the Neutrogena one because at least reef aware sounds like they're trying slightly less hard to deceive me. Oxybenzone. Checked Banana Boat. Oxybenzone. Hawaiian Tropic. Oxybenzone. Spent two more hours searching before finding Thinksport. This should not require two hours of specialist research to buy a sunscreen for a beach holiday.",
        "Score": 83,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_bk5q2d",
        "Timestamp": "18/01/2024 11:45",
        "Text": "the search engine results are part of the problem. SEO optimised greenwashing shows up first because those brands have marketing budgets. genuinely safe brands are buried",
        "Score": 345,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_cl9r6e",
        "Timestamp": "18/01/2024 13:10",
        "Text": "'reef aware' is a new one. lower claim level than 'reef safe' or 'reef friendly'. I assume the brand lawyers decided it was defensible because it doesn't actually claim anything. just awareness. vibes.",
        "Score": 423,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_dm3s9f",
        "Timestamp": "19/01/2024 8:35",
        "Text": "The two hour research burden is the actual intended outcome btw. Most people give up. They buy the thing with the nice label. The friction in finding genuinely safe products protects the greenwashing brands' market share.",
        "Score": 267,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_l2u7ez",
        "Timestamp": "06/04/2024 9:05",
        "Text": "The ACCC case against Edgewell Personal Care for Banana Boat and Hawaiian Tropic reef friendly claims is being watched closely by a lot of consumer advocates. If the ACCC wins on the reef friendly label specifically, it creates precedent that could affect every brand making similar claims. The interesting question is whether a win by the ACCC leads to brands just quietly dropping the claim, or whether it leads to actual standards being developed so the claim can be made legitimately. My fear is the former - brands drop 'reef friendly', replace it with 'ocean conscious' or some other novel unregulated phrase, and we're back to square one.",
        "Score": 74,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_en6t4g",
        "Timestamp": "06/04/2024 10:20",
        "Text": "'ocean conscious' is already a phrase several brands use and it means absolutely nothing. you're right, the whack-a-mole problem is real",
        "Score": 289,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_fo9u5h",
        "Timestamp": "06/04/2024 11:45",
        "Text": "This is why I keep saying ingredient bans are the only solution. If oxybenzone and octinoxate are banned there's nothing to mislabel. The label problem disappears.",
        "Score": 356,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_gp2v8i",
        "Timestamp": "07/04/2024 8:50",
        "Text": "The ACCC also have power to seek injunctions requiring corrective advertising. So even if Edgewell just drops the label going forward, corrective advertising orders would require them to actively tell consumers they were misled. That's the part I want to see.",
        "Score": 178,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_m5v1fa",
        "Timestamp": "22/07/2024 10:15",
        "Text": "Comparison of how eco claims are regulated: EU vs Australia vs US, specifically for sunscreen. EU: Green Claims Directive (when fully implemented) will require environmental claims to be substantiated before use with independent verification. Australia: Australian Consumer Law prohibits misleading conduct - the ACCC interprets this to cover unsubstantiated environmental claims. The Edgewell Federal Court proceedings are an example of this in action. US: The FTC Green Guides suggest companies should be able to substantiate claims but these are guidelines not laws and enforcement is rare. Summary: EU is ahead on framework, Australia is ahead on enforcement, US is behind on both. Hawaii's oxybenzone ban is the actual gold standard.",
        "Score": 82,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_hq7w6j",
        "Timestamp": "22/07/2024 11:30",
        "Text": "good breakdown. Australia ahead on enforcement is interesting because the ACL gives them real teeth. The US FTC guidelines are essentially voluntary compliance and we can see how that's gone",
        "Score": 212,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ir1x3k",
        "Timestamp": "22/07/2024 12:55",
        "Text": "the EU green claims directive is going to be interesting to watch. if it gets properly implemented it could set a global standard because brands selling into the EU market will have to comply and then just use the same formulations and claims everywhere",
        "Score": 189,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_js4y9l",
        "Timestamp": "23/07/2024 8:25",
        "Text": "Meanwhile I'm still standing in the pharmacy aisle with three products all saying reef friendly and no way to know which if any are actually true without googling every ingredient on my phone. We've been having this policy conversation for years and I still can't buy sunscreen without doing a chemistry degree first.",
        "Score": 434,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_n8w4gb",
        "Timestamp": "15/09/2024 9:35",
        "Text": "I think about the cumulative deception here and it genuinely upsets me. How many millions of people in Australia alone have bought Banana Boat or Hawaiian Tropic reef friendly products thinking they were protecting the reef? How many GBR snorkelling trips where tourists felt good about their sunscreen choice while applying something with oxybenzone? The scale of the misdirection is enormous. And it's not like this was a difficult claim to investigate - the science on oxybenzone has been clear since 2016. These brands had nearly a decade to get their formulations right or remove the claim. They chose neither. They chose profit.",
        "Score": 89,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_kt8z5m",
        "Timestamp": "15/09/2024 10:50",
        "Text": "2016 to 2025 is nine years of the science being available and the label persisting. that's not negligence, that's a choice",
        "Score": 478,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_lu2a7n",
        "Timestamp": "15/09/2024 12:15",
        "Text": "The thing about these companies is they do have R&D departments. They do have regulatory affairs teams. They knew. The decision to keep the claim was a business decision made by people who calculated the benefit outweighed the risk.",
        "Score": 312,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_mv6b3o",
        "Timestamp": "16/09/2024 8:40",
        "Text": "Edgewell Personal Care is a $2 billion company. The fine they might get from the ACCC case will be a rounding error in their annual accounts unless the penalties are genuinely substantial. This is why I'm skeptical enforcement alone changes anything.",
        "Score": 223,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_o1x6hc",
        "Timestamp": "04/12/2024 10:25",
        "Text": "Third-party certification is the only answer and I'm tired of pretending otherwise. Not brand self-certification. Not vague label claims. Actual independent testing by an accredited laboratory with published methodology and results. Stream2Sea does this - they test on coral larvae and publish the data. That's the standard. EWG verification involves actual third party review. These exist. They're not impossible. The reason mainstream brands don't do them is not that they're too hard or too expensive - it's that the products wouldn't pass. That's the whole story.",
        "Score": 87,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_nw3c8p",
        "Timestamp": "04/12/2024 11:40",
        "Text": "The products wouldn't pass is doing a lot of work in that last sentence. because it's true and it explains everything",
        "Score": 389,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ox7d2q",
        "Timestamp": "04/12/2024 13:05",
        "Text": "Mandatory third party certification with published results would end greenwashing sunscreen labels within a product cycle. Brands would reformulate because the alternative is their product failing public testing and the PR fallout from that.",
        "Score": 245,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_py4e6r",
        "Timestamp": "05/12/2024 8:30",
        "Text": "I sent emails to three Australian federal MPs asking them to support mandatory certification for reef safe claims after reading about the ACCC Edgewell case. Got two form letters and one silence. At least I did something I guess. if you care about this issue, contact your reps, it does eventually accumulate",
        "Score": 167,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    }    
]

In [9]:
# batch 5
batch_5 = [
    {
        "Post_id": "t3_f2p9kw",
        "Timestamp": "11/07/2025 8:42",
        "Text": "The ACCC Federal Court case against Edgewell Personal Care was announced this week and I've been waiting for something like this for years. For those who don't know: Edgewell makes Banana Boat and Hawaiian Tropic, both of which have been selling products labelled 'reef friendly' while containing oxybenzone and other chemical UV filters that scientists have linked to coral bleaching. The ACCC is alleging misleading conduct under Australian Consumer Law. This is a Federal Court proceeding not just a fine or a warning letter - they are serious about this. The case could set a binding precedent for what companies are allowed to claim about their products and the environment. Finally.",
        "Score": 94,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_gq3r7f",
        "Timestamp": "11/07/2025 10:05",
        "Text": "been following the ACCC's work on greenwashing for a while and this is their biggest consumer-facing case yet. they've gone after vague sustainability claims before but a specific reef friendly label on a product sold near actual reefs is a very strong case",
        "Score": 287,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_hr7s2g",
        "Timestamp": "11/07/2025 11:30",
        "Text": "I used Banana Boat reef friendly on my kids at the Whitsundays last Christmas. I specifically bought that one. I feel sick.",
        "Score": 445,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_is1t9h",
        "Timestamp": "11/07/2025 13:15",
        "Text": "Edgewell Personal Care had revenues of over two billion dollars. The penalty the ACCC can seek under Australian Consumer Law can be up to the greater of $50 million, three times the benefit obtained, or 30% of adjusted turnover during the breach period. It could be a very big number.",
        "Score": 312,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_g5q1mr",
        "Timestamp": "12/07/2025 9:18",
        "Text": "Can someone explain what the ACCC Edgewell case actually means legally? I keep seeing it mentioned but I don't fully understand the difference between this and just getting fined.",
        "Score": 71,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_jt4u5i",
        "Timestamp": "12/07/2025 10:35",
        "Text": "Federal Court proceedings means the ACCC is asking a judge to make binding legal orders. That can include civil penalties (fines), injunctions stopping them from using the label, and corrective advertising orders making them tell consumers they were misled. Much more consequential than a compliance notice.",
        "Score": 267,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ku8v3j",
        "Timestamp": "12/07/2025 12:00",
        "Text": "also discovery in Federal Court proceedings could surface internal documents showing what Edgewell actually knew and when. that could be the most damaging part for the brand reputation",
        "Score": 389,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_lv2w8k",
        "Timestamp": "12/07/2025 13:45",
        "Text": "Corrective advertising orders. That's what I want. Make them run ads saying 'we told you our products were reef friendly and that was false'. Let's see how their brand survives that.",
        "Score": 456,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_h8r4ns",
        "Timestamp": "13/07/2025 8:55",
        "Text": "Hot take: the ACCC case against Edgewell is long overdue but I'm slightly worried it will be used as proof that the system works when the system has been failing for nearly a decade. The oxybenzone research came out in 2016. Hawaii banned it in 2018. The ACCC is filing Federal Court proceedings in 2025. That's nine years of consumers buying Banana Boat and Hawaiian Tropic reef friendly products in good faith. Nine years of chemical UV filters going into reef systems near tourist sites. A court case in 2025 doesn't undo any of that. It's necessary but let's not act like it's a triumph.",
        "Score": 78,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_mw6x4l",
        "Timestamp": "13/07/2025 10:10",
        "Text": "this is the right take. glad the ACCC acted. also furious it took this long. both things are true",
        "Score": 334,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_nx9y2m",
        "Timestamp": "13/07/2025 11:35",
        "Text": "Nine years of knowingly misleading eco-conscious consumers who were specifically trying to protect the reef. that detail should be part of every penalty calculation",
        "Score": 278,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_oy3z7n",
        "Timestamp": "14/07/2025 8:20",
        "Text": "imagine being the person at Edgewell whose job was to approve putting 'reef friendly' on the label. imagine waking up this week.",
        "Score": 423,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_i1s6ot",
        "Timestamp": "14/07/2025 9:30",
        "Text": "What I keep coming back to with the ACCC v Edgewell case is how brazen it was. This wasn't a subtle claim buried in fine print. Banana Boat and Hawaiian Tropic put 'reef friendly' front and centre on products that are literally sold at beach shops near the Great Barrier Reef. They targeted the most environmentally aware consumers in the most environmentally sensitive locations. I've been to shops in Cairns and Port Douglas that had these products on prominent reef-safe displays. The geographic targeting of that deception is particularly galling. I really hope the ACCC asks for the maximum available penalty.",
        "Score": 86,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_pz5a3o",
        "Timestamp": "14/07/2025 10:45",
        "Text": "the Cairns and Whitsundays angle is so important. this wasn't just beach sunscreen sold nationally - it was specifically positioned at GBR tourist markets where people were making choices specifically about reef protection",
        "Score": 245,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_qa8b6p",
        "Timestamp": "14/07/2025 12:20",
        "Text": "Stream2Sea and Raw Elements have been doing actual reef safe formulation from the start. tiny companies with real science behind their products. meanwhile Edgewell with 2 billion in revenue just... wrote 'reef friendly' on the label. remarkable industry.",
        "Score": 312,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_j4t8pq",
        "Timestamp": "15/07/2025 8:48",
        "Text": "Reading the ACCC media release on the Edgewell Federal Court proceedings and one thing stands out: they specifically call out that the reef friendly claim was made on products containing ingredients that scientific research has associated with harm to coral reef ecosystems. The ACCC isn't just saying the claim was unsubstantiated - they're saying the evidence points the other way. This is a much stronger legal position than 'we can't prove it's true'. They're arguing 'the evidence suggests it's false'. That's the basis for serious penalties if they win.",
        "Score": 81,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_rb2c4q",
        "Timestamp": "15/07/2025 10:00",
        "Text": "the 'evidence suggests it's false not just unsubstantiated' framing is legally significant. the ACCC has done their homework here",
        "Score": 198,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_sc6d9r",
        "Timestamp": "15/07/2025 11:25",
        "Text": "Edgewell's defence will probably be that reef friendly doesn't mean reef safe and that consumers understand it as aspirational marketing not a scientific claim. watch for that argument and watch how it goes down in court.",
        "Score": 267,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_td9e3s",
        "Timestamp": "15/07/2025 13:10",
        "Text": "'reef friendly is aspirational marketing not a scientific claim' is going to be the funniest and most enraging defence I've ever read in a court document if it comes to that",
        "Score": 489,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_k7u2qr",
        "Timestamp": "16/07/2025 9:05",
        "Text": "I've spent the past few days going through all my sunscreen and checking ingredients after hearing about the ACCC Edgewell case. Found three products with oxybenzone or octinoxate in them, two of which had environmental claims on the front. All three are gone. Replaced with Thinksport SPF 50 for everyday use and Raw Elements for swimming. I know this is exactly the kind of consumer behaviour change the ACCC case is meant to drive. But I also know most people won't do this research. Regulation needs to catch up so that consumer laziness doesn't mean reef damage.",
        "Score": 68,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_ue3f7t",
        "Timestamp": "16/07/2025 10:20",
        "Text": "Thinksport and Raw Elements mentioned in the same breath again. these brands keep coming up every time people actually research this. probably because they're actually what they say they are",
        "Score": 223,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_vf7g2u",
        "Timestamp": "16/07/2025 11:45",
        "Text": "three products with oxybenzone and two had eco claims on the front. that ratio tracks with my experience. the products most likely to have misleading claims are the ones most likely to contain the bad ingredients",
        "Score": 178,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_wg1h8v",
        "Timestamp": "17/07/2025 8:30",
        "Text": "My local pharmacy has already removed the Banana Boat reef friendly products from their reef safe display since the ACCC announcement. Small win. Real change.",
        "Score": 367,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_l9v5rs",
        "Timestamp": "17/07/2025 9:40",
        "Text": "I want to be clear about something: this is not just about sunscreen. The ACCC v Edgewell Personal Care case is a test of whether Australian consumer law can hold companies accountable for environmental claims on any product. If the ACCC wins this case on a reef friendly sunscreen label, the precedent flows through to every product making unsubstantiated environmental claims - cleaning products, packaging, food, clothing. That's why this case matters beyond the coral reef issue. It's about whether 'eco-friendly' and 'sustainable' and 'natural' labels have to mean something or whether brands can just use them as marketing decoration.",
        "Score": 91,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_xh4i3w",
        "Timestamp": "17/07/2025 10:55",
        "Text": "the precedential value is the biggest deal here. ACL is broad enough that a win on reef friendly sunscreen affects the entire greenwashing landscape in Australia",
        "Score": 289,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_yi8j7x",
        "Timestamp": "17/07/2025 12:20",
        "Text": "companies watching this case closely include every brand that has put a leaf, a wave, or the word eco on a product in the last decade. which is basically all of them",
        "Score": 412,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_zj2k4y",
        "Timestamp": "18/07/2025 8:45",
        "Text": "sent the ACCC media release to every member of my family and told them to check their sunscreens. mum found two Hawaiian Tropic reef friendly bottles in the holiday kit. they're in the bin.",
        "Score": 234,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_m2w8st",
        "Timestamp": "18/07/2025 9:50",
        "Text": "What I find interesting about the timing of the ACCC Federal Court proceedings against Edgewell is that it comes after years of the ACCC issuing warnings and guidance on greenwashing without taking major enforcement action. In 2023 they did a greenwashing sweep and sent warning letters. In 2024 they announced they were prioritising environmental claims enforcement. Now in July 2025 they've gone to Federal Court. This escalation pattern suggests they've been building the case and gathering evidence for a while. The Edgewell case isn't a snap reaction to a complaint - it's a deliberate enforcement priority being executed. That gives me more confidence they'll win.",
        "Score": 76,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_ak5l2z",
        "Timestamp": "18/07/2025 11:05",
        "Text": "the ACCC greenwashing sweep in 2023 put every company on notice. Edgewell didn't change the label after that. that decision is going to be very hard to defend in court",
        "Score": 345,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_bl9m6a",
        "Timestamp": "18/07/2025 12:30",
        "Text": "knowing that the ACCC had flagged greenwashing as a priority and keeping the reef friendly label anyway is either arrogance or a calculated bet that they wouldn't be specifically targeted. neither is a good look",
        "Score": 267,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_cm3n9b",
        "Timestamp": "19/07/2025 8:15",
        "Text": "honestly embarrassed it took me this long to learn what oxybenzone is. the ACCC case made me actually look it up properly. can't believe how widespread it is and how clearly it damages coral",
        "Score": 189,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_n6x1tu",
        "Timestamp": "19/07/2025 9:25",
        "Text": "The Edgewell ACCC case has got me thinking about the business model of greenwashing. The calculation must have been: premium pricing on reef friendly label increases revenue by X. Probability of significant legal consequence times expected penalty = Y. As long as X is greater than Y, keep the label. What changes that calculation is making Y very large - either through much bigger penalties or through reputational damage that costs more than the premium pricing gained. The ACCC case could shift both sides of the equation. Higher penalties AND massive brand damage. Let's hope the Federal Court delivers on both.",
        "Score": 83,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_dn7o4c",
        "Timestamp": "19/07/2025 10:40",
        "Text": "this is exactly the right frame. greenwashing persists because the expected cost is too low. the only deterrent is making the cost high enough that no expected revenue premium justifies the risk",
        "Score": 312,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_eo1p8d",
        "Timestamp": "19/07/2025 12:05",
        "Text": "the reputational cost is already happening regardless of the court outcome. Banana Boat and Hawaiian Tropic are trending for all the wrong reasons this week. brand damage is real and immediate",
        "Score": 234,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_fp4q3e",
        "Timestamp": "20/07/2025 8:35",
        "Text": "Banana Boat: helping you feel good about destroying coral reefs since whenever they added that label. inspiring stuff.",
        "Score": 478,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_o8y3uv",
        "Timestamp": "20/07/2025 9:45",
        "Text": "I work in marine tourism and we've been trying to get suppliers to stop stocking Banana Boat reef friendly for two years. They kept saying consumers ask for it by name and the sales are too good to justify pulling it. The ACCC Federal Court proceedings have changed that conversation overnight. Our main supplier called us on Monday to say they're reviewing their range and will be pulling products with unsubstantiated reef claims. That's the market response working as it should - legal risk changing commercial behaviour faster than ethical arguments could. I'll take it.",
        "Score": 87,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_gq6r5f",
        "Timestamp": "20/07/2025 11:00",
        "Text": "the legal risk changing commercial behaviour point is key. suppliers don't want to be seen stocking products at the centre of an ACCC Federal Court case. the case is already doing work before any verdict",
        "Score": 198,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_hr9s8g",
        "Timestamp": "20/07/2025 12:25",
        "Text": "This is how market accountability is supposed to work. Legal action creates liability risk. Liability risk changes retailer behaviour. Retailer behaviour changes what's available to consumers. The ACCC case is already having effects before it's been heard.",
        "Score": 267,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_is3t2h",
        "Timestamp": "21/07/2025 8:50",
        "Text": "my dive shop pulled all Hawaiian Tropic from the shelves this week. replaced with stream2sea and a couple of other actually mineral based products. small dive shop in the whitsundays but still",
        "Score": 312,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_p1z5wv",
        "Timestamp": "21/07/2025 9:55",
        "Text": "Petition for whoever is handling the ACCC v Edgewell proceedings to get a lifetime supply of good reef-safe sunscreen as a thank you. Genuinely heroic regulatory work. I know that's silly but I've been angry about this specific issue for so long that seeing Federal Court proceedings actually filed feels like a milestone. The years of advocacy from marine scientists, environmental groups, and ordinary consumers who just wanted honest labels finally resulted in something. This case existing is itself a form of accountability even before the verdict.",
        "Score": 69,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_jt7u6i",
        "Timestamp": "21/07/2025 11:10",
        "Text": "the years of marine scientists raising the alarm and being ignored and then seeing an ACCC Federal Court case filed must feel like something. finally being heard after nearly a decade",
        "Score": 245,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_ku1v4j",
        "Timestamp": "21/07/2025 12:35",
        "Text": "Stream2Sea was built by a marine toxicologist specifically because the greenwashing was so bad and nothing was being done about it. the fact that the ACCC case might vindicate everything she's been saying for years is genuinely satisfying",
        "Score": 289,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_lv5w9k",
        "Timestamp": "22/07/2025 8:20",
        "Text": "Hawaiian Tropic 'reef friendly'. Love when the words directly contradict the chemistry. Chef's kiss. ACCC apparently felt the same way.",
        "Score": 467,
        "subreddit": "AnticonsumptionSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_q4a7xw",
        "Timestamp": "22/07/2025 9:30",
        "Text": "I want to use the ACCC Edgewell case to talk about what comes after. Even if the ACCC wins and Edgewell pays penalties and removes the reef friendly label, that doesn't automatically create a legal standard for what reef safe means. Other brands can still use the phrase. New brands can start using it. Without positive regulation defining the term and requiring substantiation, the next Edgewell is already happening somewhere on a shelf. The case is necessary. It's not sufficient. Australia needs Hawaii-style ingredient restrictions AND mandatory standards for reef safety claims. The court case is step one of a longer journey.",
        "Score": 75,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_mw8x3l",
        "Timestamp": "22/07/2025 10:45",
        "Text": "this is exactly right and I hope consumer advocates make exactly this argument to government after the case concludes. a win in court is a starting point not an ending point",
        "Score": 178,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_nx2y7m",
        "Timestamp": "22/07/2025 12:10",
        "Text": "Australia should look at what Hawaii did with the Reef Act. Banning specific ingredients removes the labelling problem entirely because there's nothing to mislabel. Regulation of claims is always a step behind creative marketing departments.",
        "Score": 223,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_oy6z2n",
        "Timestamp": "23/07/2025 8:40",
        "Text": "can someone who understands Australian law explain the timeline from here? how long until we'd actually see a verdict or settlement in the ACCC v Edgewell case?",
        "Score": 134,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_pz9a5o",
        "Timestamp": "23/07/2025 10:05",
        "Text": "Federal Court cases in Australia can move fast if the company decides to settle (months) or go much longer if it's contested (1-3 years typically). Given the strength of the ACCC's stated position and the PR damage already happening, a negotiated outcome with penalties and undertakings is probably more likely than a full hearing. The ACCC often accepts enforceable undertakings as part of settlements.",
        "Score": 189,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t3_r7b4yx",
        "Timestamp": "23/07/2025 9:15",
        "Text": "Every person I've spoken to about the ACCC Edgewell reef friendly case this week has had the same reaction: wait, reef friendly doesn't actually mean anything? That moment of realisation is happening at scale right now because of the media coverage of the case. However this ends legally, the public awareness outcome is already a win. Millions of Australians now know that reef friendly on a sunscreen label is a marketing term not a certified claim. That knowledge doesn't go away after the court case ends. The reputational damage to greenwashing labels is a permanent shift.",
        "Score": 82,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "post"
    },
    {
        "Post_id": "t1_qa4c8p",
        "Timestamp": "23/07/2025 11:30",
        "Text": "the consumer education effect of this case is huge. every news segment about it explains what oxybenzone is and why reef friendly is misleading. that reaches people who would never read a sustainability subreddit",
        "Score": 356,
        "subreddit": "sustainabilitySubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_rb8d3q",
        "Timestamp": "24/07/2025 8:30",
        "Text": "my mum texted me this morning asking which sunscreen to buy for her cruise to the reef. she heard about the ACCC case on the radio. would never have asked before. already a win regardless of legal outcome",
        "Score": 412,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_sc1e7r",
        "Timestamp": "24/07/2025 10:00",
        "Text": "Thinksport for her. easy recommendation. non-nano zinc oxide, EWG verified, available in Australia, actually what it says it is. tell her welcome to the reef safe side",
        "Score": 245,
        "subreddit": "ZeroWasteSubreddit",
        "content_type": "comment"
    },
    {
        "Post_id": "t1_td5f2s",
        "Timestamp": "24/07/2025 11:25",
        "Text": "The ACCC taking Edgewell to Federal Court is the regulatory moment this space needed. But let's be honest about what precedent looks like in practice: it's slow, it's expensive, and it only catches the most egregious cases. The system change we actually need is mandatory pre-approval of environmental claims with published evidence. Until that exists, the ACCC is playing whack-a-mole with a very large mallet against a very large number of moles.",
        "Score": 198,
        "subreddit": "GreenwashingSubreddit",
        "content_type": "comment"
    }
]

In [10]:
all_data = batch_1 + batch_2 + batch_3 + batch_4 + batch_5

df_synthetic = pd.DataFrame(all_data)
print(f"Shape: {df_synthetic.shape}")
df_synthetic.to_csv("../data/processed/df_sunscreen_synthetic.csv", index=False)
print("✓ Saved!")

Shape: (260, 6)
✓ Saved!


In [11]:
# check of the style of the synthetic posts and comments
df_synthetic.sample(10)

,Post_id,Timestamp,Text,Score,subreddit,content_type
119,t1_lz1c9q,18/07/2023 10:40,the tinted daily one is a sleeper hit. i use i...,389,ZeroWasteSubreddit,comment
41,t3_y1j9qk,24/07/2025 9:20,Quick reminder that the Great Barrier Reef is ...,76,sustainabilitySubreddit,post
218,t1_oy3z7n,14/07/2025 8:20,imagine being the person at Edgewell whose job...,423,AnticonsumptionSubreddit,comment
238,t3_n6x1tu,19/07/2025 9:25,The Edgewell ACCC case has got me thinking abo...,83,AnticonsumptionSubreddit,post
22,t1_qx1m6u,19/07/2025 13:55,The ACCC filing is publicly available if anyon...,142,sustainabilitySubreddit,comment
191,t3_l2u7ez,06/04/2024 9:05,The ACCC case against Edgewell Personal Care f...,74,GreenwashingSubreddit,post
158,t1_fo9u7h,08/03/2022 8:30,I called the customer service line for Banana ...,489,AnticonsumptionSubreddit,comment
75,t1_td3i5y,12/07/2024 10:30,the alternate names thing is so manipulative. ...,356,AnticonsumptionSubreddit,comment
140,t1_bp8s3h,09/08/2024 11:20,this works for lots of things. switched cleani...,312,AnticonsumptionSubreddit,comment
255,t3_r7b4yx,23/07/2025 9:15,Every person I've spoken to about the ACCC Edg...,82,sustainabilitySubreddit,post


## 3. Dataset Assembly: Real + Synthetic

The augmented sunscreen dataset combines:
- **8 real Reddit documents** from the greenwashing subset
  (identified via keyword filtering)
- **260 synthetic Reddit documents** generated via Claude API,
  covering 5 thematic batches:
  1. ACCC Federal Court case against Edgewell Personal Care
  2. Oxybenzone and octinoxate science and reef damage
  3. Genuinely reef-safe brand recommendations
  4. Consumer frustration with misleading reef-safe labels
  5. Sarcasm and irony around greenwashing claims

**Total: 268 documents**

Synthetic data was generated to match the style, tone and structure
of real Reddit posts and comments, including informal language,
varying length, and realistic Reddit scores. Brand mentions cover
both greenwashing brands (Banana Boat, Hawaiian Tropic, Neutrogena,
Coppertone) and genuinely reef-safe alternatives (Stream2Sea,
Raw Elements, Thinksport, Badger Balm, Sun Bum).

The synthetic data does not include a `text_clean` column -
preprocessing will be applied in the next step before
sentiment analysis and NER.

In [12]:
df_real = pd.read_csv("../data/processed/df_sunscreen_real.csv")
df_augmented = pd.concat([df_real, df_synthetic], ignore_index=True)

print(f"Real: {len(df_real)}")
print(f"Synthetic: {len(df_synthetic)}")
print(f"Total: {len(df_augmented)}")

df_augmented.to_csv("../data/processed/df_sunscreen_augmented.csv", index=False)
print("✓ Saved!")

Real: 8
Synthetic: 260
Total: 268
✓ Saved!
